# Gene Selection and Non-Usable Genes

- **Purpose:** Inspect genes that cannot be mapped between the datasets and document the gene-selection rationale.
- **Inputs:** `data/raw/geneAnno.csv`, `data/raw/SCP542_UMIcount_data.txt`.
- **Outputs:** Printed summaries and tables shown in the notebook.
- **Status:** Exploratory; not part of the final pipeline.


Here i wanted to add a exploration of the currently not used genes. The genes we will be taking a closer look at are the cell line genes that are not included in the simple intersection with the tumor genes. The main reason why i decided to check these genes once more is that I have detected some discrepancies between the tumor and cell line cells, although both of the datasets use the gh38 annotation but seemingly different versions. 

ACRC/CGNA gene is one such example with differing gene annotations. But the gene_annotation file includes only one of the gene_symbols. The cell line data contains only the gene_symbol. The tumor data contains both gene_symbols and gene_ids. The gene_annotation file includes a 1 to 1 map of gene_symbol gene_id.

Including the gene_ids to the cell lines we still obtain the same intersection as if we only considered the gene_symbols. The main ideas to fix these discrepencies are: 
1. Find a fitting gene_annotation file for each of the datasets (tumor should be fine as GCNA in gene_anno but cell line data has ACRC name that is not in the gene_anno). Then include the ensembl_id of the cell lin genes and use these to matcch the genes more accurately.
2. Do it manually (or use a libaray) to match the gene_annotations across across versions. Though i am not sure how well that will work it. 


## Generate a final gene set that will be used

In [1]:
import pandas as pd
import scanpy as sc
import anndata as ad
import numpy as np
import os

from scipy import sparse

import h5py

In [2]:
DATA_DIR = os.path.join("..", "..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

tumor_file = os.path.join(DATA_DIR, "raw", "GSE178341_crc10x_full_c295v4_submit.h5")
cell_line_file = os.path.join(DATA_DIR, "raw", "SCP542_UMIcount_data.txt")
gene_anno_file = os.path.join(DATA_DIR, "raw", "geneAnno.csv")

In [3]:
def check_df(df):
    print(f"Gene symbol (index) unique?: {df.index.is_unique}")
    print(f"Gene id unique?: {df['gene_id'].is_unique}")
    if df.columns.__contains__("gene_id_base"):
        print(f"Gene id base unique?: {df['gene_id_base'].is_unique}")
    if df.columns.__contains__("gene_id"):
        print(f"Gene symbol (column) unique?: {df['gene_id'].is_unique}")

# Prepare cell line genes

In [4]:
df_cell_line = pd.read_csv(cell_line_file, sep="\t", usecols=[0], index_col=0, skiprows=2)
df_cell_line.index.name = "gene_symbol"
df_cell_line.head()

""
gene_symbol
A1BG
A1BG-AS1
A1CF
A2M
A2M-AS1


In [5]:
gene_anno = pd.read_csv(gene_anno_file, usecols=["ensembl_gene_id", "external_gene_name"])
gene_anno["ensembl_gene_id_base"] = gene_anno["ensembl_gene_id"].str.split(".").str[0]

symbol_to_ensembl = gene_anno.set_index("external_gene_name")["ensembl_gene_id_base"].to_dict()
ensembl_to_symbol = gene_anno.set_index("ensembl_gene_id_base")["external_gene_name"].to_dict()

In [6]:
df_cell_line["gene_id"] = df_cell_line.index.map(
    lambda x: symbol_to_ensembl.get(x)
)

df_cell_line.dropna(inplace=True)
df_cell_line.head()

,gene_id
gene_symbol,
A1BG,ENSG00000121410
A1BG-AS1,ENSG00000268895
A1CF,ENSG00000148584
A2M,ENSG00000175899
A2ML1,ENSG00000166535


In [7]:
check_df(df_cell_line)

Gene symbol (index) unique?: True
Gene id unique?: True
Gene symbol (column) unique?: True


# Tumor data

In [8]:
data = h5py.File(tumor_file, "r")
data["matrix"].keys()

<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>

In [9]:
data["matrix"]["features"].keys()

<KeysViewHDF5 ['_all_tag_keys', 'feature_type', 'genome', 'id', 'name']>

In [10]:
matrix_group = data["matrix"]
features = matrix_group["features"]

gene_ids = [x.decode() if isinstance(x, bytes) else x for x in features["id"][:]]
gene_names = [x.decode() if isinstance(x, bytes) else x for x in features["name"][:]]

In [11]:
df_tumor = pd.DataFrame(gene_ids, columns=["gene_id"])
df_tumor.index = gene_names
df_tumor.index.name = "gene_symbol"

In [12]:
df_tumor = df_tumor[df_tumor["gene_id"].str.startswith(("ENSG"))].copy()
df_tumor["gene_id_base"] = df_tumor["gene_id"].str.split(".").str[0]

In [13]:
df_tumor

,gene_id,gene_id_base
gene_symbol,,
RP11-34P13.3,ENSG00000243485.5_4,ENSG00000243485
FAM138A,ENSG00000237613.2_2,ENSG00000237613
OR4F5,ENSG00000186092.6_4,ENSG00000186092
RP11-34P13.7,ENSG00000238009.6_5,ENSG00000238009
RP11-34P13.8,ENSG00000239945.1_5,ENSG00000239945
...,...,...
CH507-145C22.4,ENSG00000279313.1_4,ENSG00000279313
CH507-145C22.3,ENSG00000278878.1_4,ENSG00000278878
CH507-513H4.4,ENSG00000280614.1_4,ENSG00000280614


In [14]:
df_tumor[df_tumor["gene_id_base"].duplicated(keep=False)].sort_values("gene_id")

,gene_id,gene_id_base
gene_symbol,,
CD99,ENSG00000002586.19_4,ENSG00000002586
CD99,ENSG00000002586.19_4_PAR_Y,ENSG00000002586
VAMP7,ENSG00000124333.15_2,ENSG00000124333
VAMP7,ENSG00000124333.15_2_PAR_Y,ENSG00000124333
IL9R,ENSG00000124334.17_4,ENSG00000124334
IL9R,ENSG00000124334.17_4_PAR_Y,ENSG00000124334
PPP2R3B,ENSG00000167393.17_3,ENSG00000167393
PPP2R3B,ENSG00000167393.17_3_PAR_Y,ENSG00000167393
SPRY3,ENSG00000168939.11_2,ENSG00000168939


In [15]:
df_tumor = df_tumor[~df_tumor["gene_id_base"].duplicated(keep="first")].copy()
check_df(df_tumor)

Gene symbol (index) unique?: False
Gene id unique?: True
Gene id base unique?: True
Gene symbol (column) unique?: True


In [16]:
df_tumor["gene_id"] = df_tumor["gene_id_base"].copy()
df_tumor = df_tumor.drop("gene_id_base", axis=1)
df_tumor.head()

,gene_id
gene_symbol,
RP11-34P13.3,ENSG00000243485
FAM138A,ENSG00000237613
OR4F5,ENSG00000186092
RP11-34P13.7,ENSG00000238009
RP11-34P13.8,ENSG00000239945


In [17]:
df_cell_line.head(3)

,gene_id
gene_symbol,
A1BG,ENSG00000121410
A1BG-AS1,ENSG00000268895
A1CF,ENSG00000148584


# Combine the genes between the two datasets

In [18]:
check_df(df_cell_line)

Gene symbol (index) unique?: True
Gene id unique?: True
Gene symbol (column) unique?: True


In [19]:
check_df(df_tumor)

Gene symbol (index) unique?: False
Gene id unique?: True
Gene symbol (column) unique?: True


In [20]:
# use the unique gene ids to match cell line genes to tumor genes
df_combined = df_cell_line.merge(
    df_tumor,
    on="gene_id",
    how="inner",
)
df_combined.head(3)

,gene_id
0,ENSG00000121410
1,ENSG00000268895
2,ENSG00000148584


In [21]:
df_combined["gene_symbol"] = df_combined["gene_id"].map(
    lambda x: ensembl_to_symbol.get(x)
)
df_combined.head(3)

,gene_id,gene_symbol
0,ENSG00000121410,A1BG
1,ENSG00000268895,A1BG-AS1
2,ENSG00000148584,A1CF


In [22]:
check_df(df_combined)

Gene symbol (index) unique?: True
Gene id unique?: True
Gene symbol (column) unique?: True
